In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import sklearn.metrics as metrics
import sys
from pathlib import Path
import datetime
import polars as pl

sys.path.append(str(Path("..").resolve()))
from config import DATA_DATASET

In [ ]:
df = pl.read_csv(DATA_DATASET / "vessel_weekly_features.csv")
df[:10]

In [ ]:
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42, max_features="sqrt", class_weight="balanced")

In [ ]:
print(df["timestamp"].head(5))
print(df["timestamp"].dtype)

In [ ]:
feature_groups = [
    "mean_moving_speed",
    "sog_p10",
    "sog_median",
    "sog_p90",
    "frac_time_slow",
    "frac_time_fast",
    "heading_cog_diff_mean",
    "fishing_ratio",
    "anchor_ratio",
    "underway_engine_ratio",
    "moored_ratio",
    "length",
    "width",
    "max_draught",
    "draught_variability",
    "length_beam_ratio",
    "draught_length_ratio",
    "bbox_area",
    "n_pings",
    "mean_ping_interval_seconds",
]

removed = [
    "CargoX_ratio",
    "CargoY_ratio",
    "CargoZ_ratio",
    "CargoOS_ratio",
    "CargoReserved_ratio",
    "rot_mean_abs",
    "rot_std",
]

df = df.with_columns(pl.col("timestamp").str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M:%S.%6f"))

train_df = df.filter(pl.col("timestamp") < datetime.datetime(2025, 8, 1))


X_train = train_df[feature_groups]
y_train = train_df["ship_type"]

In [ ]:
rf_classifier.fit(X_train, y_train)

In [ ]:
importances = rf_classifier.feature_importances_

feat_importance = pd.Series(importances, index=feature_groups).sort_values(ascending=False)
print(feat_importance)

In [ ]:
i = 0
reports = {}
while True:
    starttime = datetime.datetime(2025, 8, 1) + datetime.timedelta(days=30 * i)
    endtime = starttime + datetime.timedelta(days=30)
    test_df = df.filter(pl.col("timestamp").is_between(starttime, endtime, closed="left"))
    i += 1
    if len(test_df) == 0:
        break
    X_test = test_df[feature_groups]
    y_test = test_df["ship_type"]

    y_pred = rf_classifier.predict(X_test)
    report = metrics.classification_report(y_test, y_pred, output_dict=True)  # <-- dict
    reports[starttime.strftime("%Y-%m")] = report

In [ ]:
reports.keys()
reports["2025-08"]

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

classes = ["Cargo", "Tanker", "Fishing", "Tug", "Passenger", "macro avg", "weighted avg"]
months = list(reports.keys())

fig, ax = plt.subplots(figsize=(10, 5))
for cls in classes:
    f1_scores = []
    for month in months:
        report = reports[month]
        f1 = report.get(cls, {}).get("recall", None)
        f1_scores.append(f1)
    ax.plot(months, f1_scores, marker="o", label=cls)

ax.set_xlabel("Month")
ax.set_ylabel("F1 Score")
ax.set_title("Per-class F1 Score over time (temporal drift)")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
classes = ["macro avg", "weighted avg"]
months = list(reports.keys())

fig, ax = plt.subplots(figsize=(10, 5))

acc = []
for month in months:
    report = reports[month]
    f1 = report["accuracy"]
    acc.append(f1)
ax.plot(months, f1_scores, marker="o")

ax.set_xlabel("Month")
ax.set_ylabel("Accuracy")
ax.set_title("Per-class F1 Score over time (temporal drift)")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
print(reports[9])